**AIM**:
To solve any integer programming problem using the Branch and Bound method in Python.

PROCEDURE:
1. Formulate the integer programming problem.
2. Implement the Branch and Bound method in Python.
3. Solve the problem using the implemented method.
4. Verify the solution.

In [ ]:
import numpy as np
from scipy.optimize import linprog
from queue import Queue

# Function to solve the linear programming relaxation
def solve_lp(c, A, b, bounds):
    res = linprog(c, A_ub=A, b_ub=b, bounds=bounds, method='highs')
    return res

# Branch and Bound method
def branch_and_bound(c, A, b, bounds):
    Q = Queue()
    Q.put((c, A, b, bounds))
    best_solution = None
    best_value = float('-inf')

    while not Q.empty():
        current_problem = Q.get()
        res = solve_lp(*current_problem)

        if res.success and -res.fun > best_value:
            solution = res.x
            # Check if solution is integer
            if all(np.isclose(solution, np.round(solution))):
                value = -res.fun  # Objective function value
                if value > best_value:
                    best_value = value
                    best_solution = solution
            else:
                # Branching
                for i in range(len(solution)):
                    if not np.isclose(solution[i], np.round(solution[i])):
                        lower_bounds = list(current_problem[3])
                        upper_bounds = list(current_problem[3])

                        # Branch: x_i <= floor(solution[i])
                        lower_bounds[i] = (lower_bounds[i][0], np.floor(solution[i]))
                        Q.put((current_problem[0], current_problem[1], current_problem[2], lower_bounds))

                        # Branch: x_i >= ceil(solution[i])
                        upper_bounds[i] = (np.ceil(solution[i]), upper_bounds[i][1])
                        Q.put((current_problem[0], current_problem[1], current_problem[2], upper_bounds))
                        break

    return best_solution, best_value

# Example usage
c = [-4, -3]  # Coefficients for the objective function (maximize)
A = [[2, 1], [1, 2]]  # Coefficients for the constraints
b = [8, 6]  # RHS values for the constraints
bounds = [(0, None), (0, None)]  # Bounds for the variables

# Solve the integer programming problem
solution, value = branch_and_bound(c, A, b, bounds)
print(f"Optimal solution: {solution}")
print(f"Optimal value: {value}")


Optimal solution: [ 4. -0.]
Optimal value: 16.0
